# ReverseDB — End-to-End Demo Notebook

This notebook is a **self-contained demo** of the full ReverseDB pipeline.
No Oracle connection or package installation is required — every module is
defined inline and the pipeline runs against a hand-crafted sample schema.

### Pipeline overview
```
Sample data (TableInfo / TriggerInfo / ProcedureInfo)
       │
       ├─► 1. Classification Rules     (heuristic scoring)
       ├─► 2. Table Classifier         (picks best category per table)
       ├─► 3. Document Builder         (assembles the report data model)
       └─► 4. Markdown Writer          (renders the final .md report)
```

---


## 1 · Config dataclasses (`reversedb/config.py`)

Lightweight dataclasses that hold every tunable parameter.
In production these are populated from `config.yaml`; here we build them directly.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass

@dataclass
class DatabaseConfig:
    """Oracle connection parameters."""
    host: str
    port: int
    service_name: str
    username: str
    passwd: str

@dataclass
class AnalysisConfig:
    """Controls sampling, chunking, and classification thresholds."""
    schema: str
    batch_size: int = 10_000
    data_sample_pct: float = 0.1
    max_sample_rows: int = 2_000
    ref_data_max_rows: int = 10_000   # tables <= this are reference-data candidates
    param_table_max_rows: int = 1_000  # tables <= this (+ pattern match) → system param

@dataclass
class OutputConfig:
    report_file: str = 'reverse_engineered_report.md'
    log_level: str = 'INFO'

@dataclass
class AIConfig:
    provider: str = 'copilot'
    model: str = ''
    enabled: bool = False
    confidence_threshold: int = 0

print('Config dataclasses defined ✓')


## 2 · Metadata data-transfer objects (`reversedb/extractors/metadata.py`)

These three dataclasses carry the raw Oracle metadata extracted from
`ALL_TABLES`, `ALL_TAB_COLUMNS`, and `ALL_CONSTRAINTS`.


In [ ]:
from dataclasses import dataclass, field
from typing import Any

@dataclass
class ColumnInfo:
    name: str
    data_type: str
    data_length: int | None
    nullable: bool
    position: int

@dataclass
class ConstraintInfo:
    name: str
    constraint_type: str          # P=primary key, U=unique, R=FK, C=check
    columns: list[str] = field(default_factory=list)
    ref_owner: str | None = None
    ref_table: str | None = None
    ref_columns: list[str] = field(default_factory=list)

@dataclass
class TableInfo:
    name: str
    num_rows: int | None          # from ALL_TABLES statistics
    last_analyzed: Any | None
    columns: list[ColumnInfo] = field(default_factory=list)
    constraints: list[ConstraintInfo] = field(default_factory=list)

print('Metadata DTOs defined ✓')


## 3 · Source object data-transfer objects (`reversedb/extractors/source_objects.py`)

Carry trigger metadata and procedure/function/package source extracted from
`ALL_TRIGGERS` and `ALL_SOURCE`.


In [ ]:
@dataclass
class TriggerInfo:
    name: str
    table_name: str
    trigger_type: str   # BEFORE / AFTER / INSTEAD OF
    event: str          # INSERT, UPDATE, DELETE, or combinations
    status: str         # ENABLED / DISABLED
    source: str = ''

@dataclass
class ProcedureInfo:
    name: str
    obj_type: str       # PROCEDURE | FUNCTION | PACKAGE | PACKAGE BODY
    source: str = ''
    referenced_tables: list[str] = field(default_factory=list)

print('Source object DTOs defined ✓')


## 4 · Dependency extractor (`reversedb/extractors/dependencies.py`)

Queries `ALL_DEPENDENCIES` to build a directed graph of which
procedures/packages reference which tables.  In the demo we build this map manually.


In [ ]:
@dataclass
class DependencyEdge:
    from_name: str
    from_type: str   # PROCEDURE | FUNCTION | PACKAGE | TRIGGER …
    to_name: str
    to_type: str     # TABLE | VIEW | PROCEDURE …
    to_owner: str


def build_table_procedure_map(edges: list[DependencyEdge]) -> dict[str, list[str]]:
    """Map table_name → [procedure/function/package names] from dependency edges."""
    mapping: dict[str, list[str]] = {}
    for edge in edges:
        if edge.to_type == 'TABLE' and edge.from_type in (
            'PROCEDURE', 'FUNCTION', 'PACKAGE', 'PACKAGE BODY', 'TRIGGER'
        ):
            mapping.setdefault(edge.to_name, []).append(edge.from_name)
    return {tbl: list(dict.fromkeys(objs)) for tbl, objs in mapping.items()}

print('Dependency extractor defined ✓')


## 5 · Heuristic classification rules (`reversedb/classifiers/rules.py`)

Each function returns an integer score (0 = no evidence, higher = stronger evidence).
Scores from all rules are summed per category; the highest wins.


In [ ]:
import re

# ── Keyword sets ─────────────────────────────────────────────────────────
_TXN_NAME_KEYWORDS = {
    'TXN','TRANSACTION','TRANS','ORDER','PAYMENT','INVOICE',
    'RECEIPT','JOURNAL','LEDGER','POSTING','SETTLEMENT','CLEARING',
    'EVENT','LOG','AUDIT','HISTORY','ACTIVITY','ENTRY','MOVEMENT',
    'TRANSFER','DISPATCH','SHIPMENT','BOOKING',
}
_REF_NAME_KEYWORDS = {
    'TYPE','CODE','LOOKUP','REF','REFERENCE','STATUS','STATE',
    'CATEGORY','CLASS','GROUP','ENUM','LIST','REASON','FLAG',
    'MODE','PRIORITY','LEVEL','GRADE','CURRENCY','COUNTRY',
    'LANGUAGE','REGION','ZONE',
}
_MASTER_NAME_KEYWORDS = {
    'MASTER','CUSTOMER','CLIENT','ACCOUNT','PRODUCT','ITEM',
    'SUPPLIER','VENDOR','EMPLOYEE','STAFF','PARTY','PERSON',
    'ENTITY','ASSET','LOCATION','SITE','BRANCH','DEPARTMENT',
    'ORGANISATION','ORGANIZATION','CONTACT','PROFILE','MEMBER',
}
_PARAM_NAME_KEYWORDS = {
    'PARAM','PARAMETER','CONFIG','CONFIGURATION','SETTING',
    'CONTROL','OPTION','PROPERTY','PREFERENCE','SYSTEM',
    'GLOBAL','RUNTIME','RULE','POLICY',
}
_TXN_COLUMN_HINTS = {
    'STATUS','STATE','TXN_DATE','TRANS_DATE','CREATED_DATE',
    'POSTED_DATE','EFFECTIVE_DATE','VALUE_DATE','PROCESS_DATE',
    'APPROVAL','REFERENCE_NO','DOC_NO','VOUCHER',
}
_REF_COLUMN_HINTS  = {'CODE','DESCR','DESCRIPTION','SHORT_CODE','LONG_DESC'}
_PARAM_COLUMN_HINTS = {
    'PARAM_VALUE','PARAM_NAME','CONFIG_VALUE','CONFIG_KEY',
    'SETTING_VALUE','SETTING_NAME','PROP_VALUE','PROP_NAME',
}

# ── Primitive scorers ─────────────────────────────────────────────────────
def _name_score(table_name: str, kw: set) -> int:
    return 2 if set(re.split(r'[_\s]', table_name.upper())) & kw else 0

def _column_score(table: TableInfo, hints: set) -> int:
    parts: set[str] = set()
    for c in table.columns:
        parts.update(re.split(r'[_\s]', c.name.upper()))
    return len(parts & hints)

def _row_count_score_txn(n):
    if n is None: return 0
    if n >= 1_000_000: return 4
    if n >= 100_000:   return 2
    if n >= 10_000:    return 1
    return 0

def _row_count_score_ref(n, threshold):
    if n is None: return 1
    return 3 if n <= threshold else 0

def _row_count_score_param(n, threshold):
    if n is None: return 0
    return 3 if n <= threshold else 0

def _fk_referenced_score(name, fk_counts):
    c = fk_counts.get(name, 0)
    if c >= 10: return 4
    if c >= 5:  return 2
    if c >= 2:  return 1
    return 0

def _has_date_column(table: TableInfo) -> int:
    return 1 if any(
        c.data_type in ('DATE','TIMESTAMP') or 'DATE' in c.name.upper()
        for c in table.columns
    ) else 0

# ── Public composite scorers ──────────────────────────────────────────────
def score_transaction(table, fk_counts):
    return (_name_score(table.name, _TXN_NAME_KEYWORDS)
            + _column_score(table, _TXN_COLUMN_HINTS)
            + _row_count_score_txn(table.num_rows)
            + _has_date_column(table))

def score_reference(table, ref_max):
    return (_name_score(table.name, _REF_NAME_KEYWORDS)
            + _column_score(table, _REF_COLUMN_HINTS)
            + _row_count_score_ref(table.num_rows, ref_max))

def score_master(table, fk_counts):
    return (_name_score(table.name, _MASTER_NAME_KEYWORDS)
            + _fk_referenced_score(table.name, fk_counts))

def score_system_param(table, param_max):
    return (_name_score(table.name, _PARAM_NAME_KEYWORDS)
            + _column_score(table, _PARAM_COLUMN_HINTS)
            + _row_count_score_param(table.num_rows, param_max))

print('Classification rules defined ✓')


## 6 · Table classifier (`reversedb/classifiers/table_classifier.py`)

Orchestrates the scoring rules: for each table it collects scores for all four
categories, picks the highest, and resolves ties using a priority order
(more specific categories win).


In [ ]:
from enum import Enum
from dataclasses import dataclass as _dc

class TableCategory(str, Enum):
    TRANSACTION  = 'Transaction'
    REFERENCE    = 'Reference Data'
    MASTER       = 'Master Data'
    SYSTEM_PARAM = 'System Parameter'
    UNKNOWN      = 'Unknown'

@_dc
class ClassificationResult:
    table_name: str
    category: TableCategory
    scores: dict
    ai_confirmed: bool = False
    ai_override: 'TableCategory | None' = None

_PRIORITY = {
    TableCategory.SYSTEM_PARAM.value: 4,
    TableCategory.REFERENCE.value:    3,
    TableCategory.MASTER.value:       2,
    TableCategory.TRANSACTION.value:  1,
}

def _build_fk_ref_counts(tables):
    counts = {}
    for tbl in tables:
        for con in tbl.constraints:
            if con.constraint_type == 'R' and con.ref_table:
                counts[con.ref_table] = counts.get(con.ref_table, 0) + 1
    return counts

def classify_tables(tables: list, cfg: AnalysisConfig) -> list:
    """Classify every table and return a ClassificationResult for each."""
    fk_counts = _build_fk_ref_counts(tables)
    results = []
    for tbl in tables:
        scores = {
            TableCategory.TRANSACTION.value:  score_transaction(tbl, fk_counts),
            TableCategory.REFERENCE.value:    score_reference(tbl, cfg.ref_data_max_rows),
            TableCategory.MASTER.value:       score_master(tbl, fk_counts),
            TableCategory.SYSTEM_PARAM.value: score_system_param(tbl, cfg.param_table_max_rows),
        }
        best = max(scores.values())
        if best == 0:
            category = TableCategory.UNKNOWN
        else:
            best_name = max(
                (k for k, v in scores.items() if v == best),
                key=lambda k: _PRIORITY.get(k, 0),
            )
            category = TableCategory(best_name)
        results.append(ClassificationResult(table_name=tbl.name, category=category, scores=scores))
    return results

print('Table classifier defined ✓')


## 7 · Report document builder (`reversedb/reporters/document.py`)

Assembles the classification results together with column/constraint/trigger
metadata into a structured `ReportDocument` object (no formatting at this stage).


In [ ]:
import datetime

# ── Report data model ─────────────────────────────────────────────────────
@_dc
class ColumnEntry:
    name: str
    data_type: str
    nullable: bool

from dataclasses import field as _field

@_dc
class TransactionEntry:
    table_name: str
    num_rows: 'int | None'
    ai_confirmed: bool = False
    ai_note: 'str | None' = None
    columns: list = _field(default_factory=list)
    status_columns: list = _field(default_factory=list)
    date_columns: list = _field(default_factory=list)
    fk_refs: list = _field(default_factory=list)
    triggers: list = _field(default_factory=list)
    procedures: list = _field(default_factory=list)

@_dc
class ReferenceEntry:
    table_name: str
    num_rows: 'int | None'
    ai_confirmed: bool = False
    ai_note: 'str | None' = None
    columns: list = _field(default_factory=list)
    fk_consumers: list = _field(default_factory=list)
    owner: str = 'Admin / System'

@_dc
class MasterEntry:
    table_name: str
    num_rows: 'int | None'
    ai_confirmed: bool = False
    ai_note: 'str | None' = None
    primary_key: list = _field(default_factory=list)
    columns: list = _field(default_factory=list)
    fk_consumers: list = _field(default_factory=list)
    procedures: list = _field(default_factory=list)

@_dc
class SystemParamEntry:
    table_name: str
    num_rows: 'int | None'
    ai_confirmed: bool = False
    ai_note: 'str | None' = None
    key_columns: list = _field(default_factory=list)
    value_columns: list = _field(default_factory=list)
    procedures: list = _field(default_factory=list)

@_dc
class ReportDocument:
    schema: str
    generated_at: str
    total_tables: int
    total_triggers: int
    total_procedures: int
    transactions: list = _field(default_factory=list)
    references: list = _field(default_factory=list)
    masters: list = _field(default_factory=list)
    system_params: list = _field(default_factory=list)
    unknown_tables: list = _field(default_factory=list)

# ── Builder ───────────────────────────────────────────────────────────────
def build_document(schema, tables, triggers, procedures, classifications, table_proc_map):
    tables_idx  = {t.name: t for t in tables}
    triggers_by = {}
    for trg in triggers:
        triggers_by.setdefault(trg.table_name, []).append(trg.name)
    fk_consumers: dict = {}
    for tbl in tables:
        for con in tbl.constraints:
            if con.constraint_type == 'R' and con.ref_table:
                fk_consumers.setdefault(con.ref_table, []).append(tbl.name)

    doc = ReportDocument(
        schema=schema,
        generated_at=datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d %H:%M UTC'),
        total_tables=len(tables),
        total_triggers=len(triggers),
        total_procedures=len(procedures),
    )

    def _cols(tbl): return [ColumnEntry(c.name, c.data_type, c.nullable) for c in tbl.columns]
    def _ai_note(clf):
        if clf.ai_override:
            return f'AI override: heuristic `{clf.category.value}` → AI `{clf.ai_override.value}`'
        return None

    for clf in classifications:
        tbl = tables_idx.get(clf.table_name)
        if tbl is None: continue
        cat = clf.ai_override or clf.category
        note = _ai_note(clf)

        if cat == TableCategory.TRANSACTION:
            status_cols = [c.name for c in tbl.columns if any(k in c.name.upper() for k in ('STATUS','STATE','APPROVAL'))]
            date_cols   = [c.name for c in tbl.columns if c.data_type in ('DATE','TIMESTAMP') or 'DATE' in c.name.upper()]
            fk_refs     = list(dict.fromkeys(con.ref_table for con in tbl.constraints if con.constraint_type=='R' and con.ref_table))
            doc.transactions.append(TransactionEntry(
                table_name=tbl.name, num_rows=tbl.num_rows,
                ai_confirmed=clf.ai_confirmed, ai_note=note,
                columns=_cols(tbl), status_columns=status_cols,
                date_columns=date_cols, fk_refs=fk_refs,
                triggers=triggers_by.get(tbl.name, []),
                procedures=table_proc_map.get(tbl.name, []),
            ))
        elif cat == TableCategory.REFERENCE:
            doc.references.append(ReferenceEntry(
                table_name=tbl.name, num_rows=tbl.num_rows,
                ai_confirmed=clf.ai_confirmed, ai_note=note,
                columns=_cols(tbl),
                fk_consumers=list(dict.fromkeys(fk_consumers.get(tbl.name,[]))),
            ))
        elif cat == TableCategory.MASTER:
            pk = next((con.columns for con in tbl.constraints if con.constraint_type=='P'), [])
            doc.masters.append(MasterEntry(
                table_name=tbl.name, num_rows=tbl.num_rows,
                ai_confirmed=clf.ai_confirmed, ai_note=note,
                primary_key=pk, columns=_cols(tbl),
                fk_consumers=list(dict.fromkeys(fk_consumers.get(tbl.name,[]))),
                procedures=table_proc_map.get(tbl.name, []),
            ))
        elif cat == TableCategory.SYSTEM_PARAM:
            key_hints = {'PARAM_NAME','CONFIG_KEY','SETTING_NAME','PROP_NAME','NAME','KEY'}
            val_hints = {'PARAM_VALUE','CONFIG_VALUE','SETTING_VALUE','PROP_VALUE','VALUE','DATA'}
            doc.system_params.append(SystemParamEntry(
                table_name=tbl.name, num_rows=tbl.num_rows,
                ai_confirmed=clf.ai_confirmed, ai_note=note,
                key_columns=[c.name for c in tbl.columns if c.name.upper() in key_hints],
                value_columns=[c.name for c in tbl.columns if c.name.upper() in val_hints],
                procedures=table_proc_map.get(tbl.name, []),
            ))
        else:
            doc.unknown_tables.append(tbl.name)
    return doc

print('Document builder defined ✓')


## 8 · Markdown writer (`reversedb/reporters/markdown_writer.py`)

Renders the `ReportDocument` into a structured Markdown file with six sections:
**A** Executive Summary · **B** Transactions · **C** Reference Data ·
**D** Master Data · **E** System Parameters · **F** Unclassified


In [ ]:
_NA = 'n/a'

def _fmt_rows(n):
    return f'{n:,}' if n is not None else 'unknown (run DBMS_STATS.GATHER_SCHEMA_STATS to refresh)'

def _bullet(items, indent=0):
    p = '  ' * indent
    return (f'{p}- _(none)_\n') if not items else ''.join(f'{p}- `{i}`\n' for i in items)

def render_markdown(doc: ReportDocument) -> str:
    lines = []
    def h(lv, txt): lines.append(f'{"#"*lv} {txt}\n')

    h(1, f'Reverse-Engineered Database Report – `{doc.schema}`')
    lines += [f'_Generated: {doc.generated_at}_\n', '---\n']

    # A. Executive summary
    h(2, 'A. Executive Summary')
    lines.append(
        f'| Item | Count |\n|---|---|\n'
        f'| Schema | `{doc.schema}` |\n'
        f'| Total tables | {doc.total_tables} |\n'
        f'| Triggers | {doc.total_triggers} |\n'
        f'| Procedures / Functions / Packages | {doc.total_procedures} |\n'
        f'| Transaction tables | {len(doc.transactions)} |\n'
        f'| Reference-data tables | {len(doc.references)} |\n'
        f'| Master-data tables | {len(doc.masters)} |\n'
        f'| System-parameter tables | {len(doc.system_params)} |\n'
        f'| Unclassified tables | {len(doc.unknown_tables)} |\n'
    )
    lines.append('\n')

    # B. Transactions
    h(2, 'B. Transaction Types')
    if not doc.transactions:
        lines.append('_No transaction tables identified._\n\n')
    for e in sorted(doc.transactions, key=lambda x: x.table_name):
        h(3, f'`{e.table_name}`')
        lines.append(f'- **Estimated rows:** {_fmt_rows(e.num_rows)}\n')
        if e.ai_confirmed: lines.append('- **AI review:** ✅ Confirmed\n')
        elif e.ai_note:    lines.append(f'- **AI review:** {e.ai_note}\n')
        lines.append('- **Columns:**\n')
        for c in e.columns:
            nf = ' _(nullable)_' if c.nullable else ''
            lines.append(f'  - `{c.name}` ({c.data_type}){nf}\n')
        lines.append(f'- **Status / state columns:** {", ".join(f"`{c}`" for c in e.status_columns) or _NA}\n')
        lines.append(f'- **Date / timestamp columns:** {", ".join(f"`{c}`" for c in e.date_columns) or _NA}\n')
        lines.append(f'- **References (FK targets):** {", ".join(f"`{t}`" for t in e.fk_refs) or _NA}\n')
        lines.append(f'- **Triggers:**\n{_bullet(e.triggers, 1)}')
        lines.append(f'- **Procedures / packages:**\n{_bullet(e.procedures, 1)}')
        lines.append('\n')

    # C. Reference data
    h(2, 'C. Reference Data')
    if not doc.references:
        lines.append('_No reference-data tables identified._\n\n')
    for e in sorted(doc.references, key=lambda x: x.table_name):
        h(3, f'`{e.table_name}`')
        lines.append(f'- **Estimated rows:** {_fmt_rows(e.num_rows)}\n')
        if e.ai_confirmed: lines.append('- **AI review:** ✅ Confirmed\n')
        elif e.ai_note:    lines.append(f'- **AI review:** {e.ai_note}\n')
        lines.append(f'- **Update ownership:** {e.owner}\n')
        lines.append(f'- **Consumed by (FK):**\n{_bullet(e.fk_consumers, 1)}')
        lines.append('- **Columns:**\n')
        for c in e.columns: lines.append(f'  - `{c.name}` ({c.data_type})\n')
        lines.append('\n')

    # D. Master data
    h(2, 'D. Master Data')
    if not doc.masters:
        lines.append('_No master-data tables identified._\n\n')
    for e in sorted(doc.masters, key=lambda x: x.table_name):
        h(3, f'`{e.table_name}`')
        lines.append(f'- **Estimated rows:** {_fmt_rows(e.num_rows)}\n')
        if e.ai_confirmed: lines.append('- **AI review:** ✅ Confirmed\n')
        elif e.ai_note:    lines.append(f'- **AI review:** {e.ai_note}\n')
        lines.append(f'- **Primary key:** {", ".join(f"`{c}`" for c in e.primary_key) or _NA}\n')
        lines.append(f'- **Referenced by (FK):**\n{_bullet(e.fk_consumers, 1)}')
        lines.append(f'- **Procedures / packages:**\n{_bullet(e.procedures, 1)}')
        lines.append('- **Columns:**\n')
        for c in e.columns:
            nf = ' _(nullable)_' if c.nullable else ''
            lines.append(f'  - `{c.name}` ({c.data_type}){nf}\n')
        lines.append('\n')

    # E. System params
    h(2, 'E. System Parameters')
    if not doc.system_params:
        lines.append('_No system-parameter tables identified._\n\n')
    for e in sorted(doc.system_params, key=lambda x: x.table_name):
        h(3, f'`{e.table_name}`')
        lines.append(f'- **Estimated rows:** {_fmt_rows(e.num_rows)}\n')
        if e.ai_confirmed: lines.append('- **AI review:** ✅ Confirmed\n')
        elif e.ai_note:    lines.append(f'- **AI review:** {e.ai_note}\n')
        lines.append(f'- **Key columns:** {", ".join(f"`{c}`" for c in e.key_columns) or _NA}\n')
        lines.append(f'- **Value columns:** {", ".join(f"`{c}`" for c in e.value_columns) or _NA}\n')
        lines.append(f'- **Procedures / packages:**\n{_bullet(e.procedures, 1)}')
        lines.append('\n')

    # F. Unclassified
    h(2, 'F. Unclassified Tables')
    if not doc.unknown_tables:
        lines.append('_All tables were classified._\n\n')
    else:
        lines.append('The following tables did not match any classification heuristic.\n\n')
        for name in sorted(doc.unknown_tables): lines.append(f'- `{name}`\n')
        lines.append('\n')

    return '\n'.join(lines)

print('Markdown writer defined ✓')


## 9 · DB layer — reference only (`reversedb/db/`)

The two DB classes below are **not executed** in this demo (no Oracle connection
is available), but the source is shown here so the notebook captures the full
codebase.

### `OracleConnector` — thin context-manager wrapper around `oracledb`
```python
with OracleConnector(cfg.database) as conn:
    cursor = conn.cursor()
```

### `ChunkedReader` — safe large-table reads
| Method | Purpose |
|---|---|
| `sample_table(table, columns, sample_pct, max_rows)` | Uses `SAMPLE(pct)` + `FETCH FIRST n ROWS ONLY` — never full-scans |
| `stream_query(sql, bind_vars)` | Yields batches via `cursor.fetchmany(batch_size)` |
| `fetch_all(sql, bind_vars)` | Convenience wrapper for bounded queries |


In [ ]:
# ── OracleConnector (source shown; not executed) ──────────────────────────
# import oracledb
# from contextlib import contextmanager
#
# class OracleConnector:
#     def __init__(self, db_cfg): self._cfg = db_cfg; self._conn = None
#     def __enter__(self): self.connect(); return self
#     def __exit__(self, *a): self.close()
#     def connect(self):
#         dsn = oracledb.makedsn(self._cfg.host, self._cfg.port, service_name=self._cfg.service_name)
#         self._conn = oracledb.connect(user=self._cfg.username, ****** dsn=dsn)
#     def close(self):
#         if self._conn: self._conn.close(); self._conn = None
#     def cursor(self): return self._conn.cursor()
#     @contextmanager
#     def managed_cursor(self):
#         cur = self.cursor()
#         try: yield cur
#         finally: cur.close()

# ── ChunkedReader (source shown; not executed) ────────────────────────────
# class ChunkedReader:
#     def __init__(self, connector, batch_size=10_000):
#         self._connector = connector; self.batch_size = batch_size
#     def sample_table(self, table, columns, sample_pct=0.1, max_rows=2_000, ...):
#         sql = f'SELECT {col_list} FROM {table} SAMPLE({sample_pct:.6f}) FETCH FIRST :max_rows ROWS ONLY'
#         with self._connector.managed_cursor() as cur:
#             cur.execute(sql, {**bind_vars, 'max_rows': max_rows})
#             return [dict(zip([d[0].lower() for d in cur.description], row)) for row in cur.fetchall()]
#     def stream_query(self, sql, bind_vars=None):
#         with self._connector.managed_cursor() as cur:
#             cur.arraysize = self.batch_size; cur.execute(sql, bind_vars or {})
#             col_names = [d[0].lower() for d in cur.description]
#             while rows := cur.fetchmany(self.batch_size): yield [dict(zip(col_names, r)) for r in rows]
#     def fetch_all(self, sql, bind_vars=None):
#         return [r for batch in self.stream_query(sql, bind_vars) for r in batch]

print('DB layer source shown (commented out — no Oracle connection needed for demo) ✓')


## 10 · AI reviewer — reference only (`reversedb/ai/`)

When `ai.enabled: true` in `config.yaml` the pipeline calls the GitHub Copilot
chat-completions API to confirm or override each heuristic classification.

```
CopilotReviewer.review(classifications)
  └─ builds a strict JSON prompt with heuristic scores
  └─ POST https://api.githubcopilot.com/chat/completions
  └─ parses {"reviews":[{confirmed, override_category}]}
  └─ applies ai_confirmed=True or ai_override=<category>
```

Set `COPILOT_API_KEY` / `GITHUB_TOKEN` env-var to enable.


In [ ]:
# AI reviewer is disabled for this demo (no API key required).
# The interface is:
#
#   from reversedb.ai.factory import get_reviewer
#   reviewer = get_reviewer(cfg.ai)          # returns CopilotReviewer
#   classifications = reviewer.review(classifications)
#
# CopilotReviewer sends all heuristic results to the Copilot API and
# updates ai_confirmed / ai_override on each ClassificationResult.

print('AI reviewer shown as reference ✓')


## 11 · Sample dataset — `LEGACY_ERP` schema

Eight representative tables covering all four classification categories.
These objects stand in for what `MetadataExtractor` would return from Oracle.


In [ ]:
# ── Tables ────────────────────────────────────────────────────────────────
orders = TableInfo('ORDERS', 5_200_000, '2025-06-01',
    columns=[
        ColumnInfo('ORDER_ID',     'NUMBER',   10,   False, 1),
        ColumnInfo('CUSTOMER_ID',  'NUMBER',   10,   False, 2),
        ColumnInfo('ORDER_DATE',   'DATE',     None, False, 3),
        ColumnInfo('STATUS',       'VARCHAR2', 20,   False, 4),
        ColumnInfo('TOTAL_AMOUNT', 'NUMBER',   None, True,  5),
        ColumnInfo('CREATED_DATE', 'DATE',     None, True,  6),
        ColumnInfo('LAST_UPDATED', 'TIMESTAMP',None, True,  7),
    ],
    constraints=[
        ConstraintInfo('PK_ORDERS',   'P', ['ORDER_ID']),
        ConstraintInfo('FK_ORD_CUST', 'R', ['CUSTOMER_ID'], ref_table='CUSTOMERS'),
        ConstraintInfo('FK_ORD_PROD', 'R', ['PRODUCT_ID'],  ref_table='PRODUCT_CATALOG'),
    ])

order_items = TableInfo('ORDER_ITEMS', 18_400_000, '2025-06-01',
    columns=[
        ColumnInfo('ITEM_ID',         'NUMBER',   10,  False, 1),
        ColumnInfo('ORDER_ID',        'NUMBER',   10,  False, 2),
        ColumnInfo('PRODUCT_ID',      'NUMBER',   10,  False, 3),
        ColumnInfo('QUANTITY',        'NUMBER',   None,False, 4),
        ColumnInfo('UNIT_PRICE',      'NUMBER',   None,False, 5),
        ColumnInfo('APPROVAL_STATUS', 'VARCHAR2', 20,  True,  6),
    ],
    constraints=[
        ConstraintInfo('PK_ITEMS',     'P', ['ITEM_ID']),
        ConstraintInfo('FK_ITEMS_ORD', 'R', ['ORDER_ID'],   ref_table='ORDERS'),
        ConstraintInfo('FK_ITEMS_PRD', 'R', ['PRODUCT_ID'], ref_table='PRODUCT_CATALOG'),
    ])

customers = TableInfo('CUSTOMERS', 950_000, '2025-06-01',
    columns=[
        ColumnInfo('CUSTOMER_ID',  'NUMBER',   10,  False, 1),
        ColumnInfo('FULL_NAME',    'VARCHAR2', 200, False, 2),
        ColumnInfo('EMAIL',        'VARCHAR2', 200, True,  3),
        ColumnInfo('COUNTRY_CODE', 'CHAR',     2,   False, 4),
        ColumnInfo('CREATED_DATE', 'DATE',     None,True,  5),
        ColumnInfo('SEGMENT',      'VARCHAR2', 50,  True,  6),
    ],
    constraints=[ConstraintInfo('PK_CUSTOMERS','P',['CUSTOMER_ID'])])

products = TableInfo('PRODUCT_CATALOG', 85_000, '2025-06-01',
    columns=[
        ColumnInfo('PRODUCT_ID',   'NUMBER',   10,  False, 1),
        ColumnInfo('PRODUCT_CODE', 'VARCHAR2', 50,  False, 2),
        ColumnInfo('DESCRIPTION',  'VARCHAR2', 500, True,  3),
        ColumnInfo('CATEGORY_ID',  'NUMBER',   10,  False, 4),
        ColumnInfo('UNIT_PRICE',   'NUMBER',   None,False, 5),
        ColumnInfo('IS_ACTIVE',    'CHAR',     1,   False, 6),
    ],
    constraints=[
        ConstraintInfo('PK_PRODUCTS','P',['PRODUCT_ID']),
        ConstraintInfo('FK_PROD_CAT','R',['CATEGORY_ID'],ref_table='CATEGORIES'),
    ])

currencies = TableInfo('CURRENCIES', 150, '2025-06-01',
    columns=[
        ColumnInfo('CURRENCY_CODE','CHAR',    3,   False,1),
        ColumnInfo('CURRENCY_NAME','VARCHAR2',100, False,2),
        ColumnInfo('SYMBOL',       'VARCHAR2',10,  True, 3),
    ],
    constraints=[ConstraintInfo('PK_CURRENCIES','P',['CURRENCY_CODE'])])

categories = TableInfo('CATEGORIES', 42, '2025-06-01',
    columns=[
        ColumnInfo('CATEGORY_ID',  'NUMBER',   10, False,1),
        ColumnInfo('CATEGORY_CODE','VARCHAR2', 20, False,2),
        ColumnInfo('DESCRIPTION',  'VARCHAR2',200, True, 3),
    ],
    constraints=[ConstraintInfo('PK_CATEGORIES','P',['CATEGORY_ID'])])

app_config = TableInfo('APP_CONFIG', 320, '2025-06-01',
    columns=[
        ColumnInfo('PARAM_NAME',   'VARCHAR2',100, False,1),
        ColumnInfo('PARAM_VALUE',  'VARCHAR2',500, True, 2),
        ColumnInfo('DESCRIPTION',  'VARCHAR2',500, True, 3),
        ColumnInfo('LAST_MODIFIED','DATE',    None,True, 4),
    ],
    constraints=[ConstraintInfo('PK_APP_CONFIG','P',['PARAM_NAME'])])

feature_flags = TableInfo('FEATURE_FLAGS', 88, '2025-06-01',
    columns=[
        ColumnInfo('FLAG_KEY','VARCHAR2',100,False,1),
        ColumnInfo('VALUE',   'VARCHAR2', 20,False,2),
        ColumnInfo('MODULE',  'VARCHAR2', 50,True, 3),
    ],
    constraints=[ConstraintInfo('PK_FLAGS','P',['FLAG_KEY'])])

ALL_TABLES = [orders, order_items, customers, products, currencies, categories, app_config, feature_flags]

# ── Triggers ───────────────────────────────────────────────────────────────
ALL_TRIGGERS = [
    TriggerInfo('TRG_ORDERS_AUDIT',  'ORDERS',      'AFTER',  'INSERT OR UPDATE OR DELETE','ENABLED'),
    TriggerInfo('TRG_ORDERS_STATUS', 'ORDERS',      'BEFORE', 'UPDATE',                   'ENABLED'),
    TriggerInfo('TRG_ITEMS_CALC',    'ORDER_ITEMS', 'BEFORE', 'INSERT OR UPDATE',          'ENABLED'),
]

# ── Procedures / packages ─────────────────────────────────────────────────
ALL_PROCEDURES = [
    ProcedureInfo('PKG_ORDER_MGMT',    'PACKAGE',      referenced_tables=['ORDERS','ORDER_ITEMS','CUSTOMERS']),
    ProcedureInfo('PKG_ORDER_MGMT',    'PACKAGE BODY', referenced_tables=[]),
    ProcedureInfo('PROC_CLOSE_ORDER',  'PROCEDURE',    referenced_tables=['ORDERS']),
    ProcedureInfo('FN_GET_TOTAL',      'FUNCTION',     referenced_tables=['ORDER_ITEMS']),
    ProcedureInfo('PKG_PRODUCT_UTILS', 'PACKAGE',      referenced_tables=['PRODUCT_CATALOG','CATEGORIES']),
]

print(f'Sample schema built: {len(ALL_TABLES)} tables, {len(ALL_TRIGGERS)} triggers, {len(ALL_PROCEDURES)} procedures ✓')


## 12 · Run the full pipeline

Execute classify → build document → render Markdown — all in-memory.


In [ ]:
# Build table → procedure map
table_proc_map: dict = {}
for proc in ALL_PROCEDURES:
    for tbl in proc.referenced_tables:
        table_proc_map.setdefault(tbl, [])
        if proc.name not in table_proc_map[tbl]:
            table_proc_map[tbl].append(proc.name)

# Classify
SCHEMA = 'LEGACY_ERP'
cfg = AnalysisConfig(schema=SCHEMA, ref_data_max_rows=10_000, param_table_max_rows=1_000)
classifications = classify_tables(ALL_TABLES, cfg)

print('Classification results:')
print(f'  {"Table":<25} {"Category"}')
print('  ' + '-'*45)
for r in classifications:
    print(f'  {r.table_name:<25} {r.category.value}')


In [ ]:
# Build document
doc = build_document(
    schema=SCHEMA,
    tables=ALL_TABLES,
    triggers=ALL_TRIGGERS,
    procedures=ALL_PROCEDURES,
    classifications=classifications,
    table_proc_map=table_proc_map,
)
print(f'Document built: {len(doc.transactions)} transaction, {len(doc.references)} reference, '
      f'{len(doc.masters)} master, {len(doc.system_params)} system-param tables ✓')


In [ ]:
# Render Markdown and display inline
report_md = render_markdown(doc)

# Write to file
output_path = 'demo_report.md'
with open(output_path, 'w', encoding='utf-8') as fh:
    fh.write(report_md)
print(f'Report written to: {output_path}')

# Pretty-print the first 80 lines as a preview
preview_lines = report_md.splitlines()[:80]
print('\n' + '='*60 + ' REPORT PREVIEW ' + '='*60)
print('\n'.join(preview_lines))
print('...(truncated — open demo_report.md for the full report)')
